# Flipkart Gridlock 2.0: V15 Perfected Graph Architecture
## Dynamic Asymmetric Loss & Environmental Risk Scaling

**Architectural Paradigm:**
This definitive pipeline discards the mathematical trap of residual spillover and returns to the physically grounded Raw Spatial Graph. To breach the final 93.0 performance threshold, it introduces two advanced micro-optimizations: an **Environmental Risk Multiplier** that scales weather impacts based on historical baseline congestion, and a **Dynamic Asymmetric Loss** function that penalizes under-predictions continuously based on the severity of the true traffic demand.

### Phase 1: Environment Initialization & Matrix Ingestion
Configures the global environment, enforces deterministic seeds, and loads the foundational Level-1 data matrices.

In [11]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

data_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in data_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train = raw_train['demand'].values
submission_index = raw_test['Index'].values

print("System initialized. Level-1 base matrices loaded.")

System initialized. Level-1 base matrices loaded.


### Phase 2: Raw Spatial Graph & Environmental Scaling
Computes the 8-node physical adjacency graph and temporal momentum vectors on raw physical traffic demand. It establishes the Out-of-Fold historical baseline and explicitly multiplies it by the categorical weather index. This interaction allows the model to mathematically recognize that extreme weather exponentially amplifies existing traffic bottlenecks while having negligible impact on empty roads.

In [2]:
def engineer_v15_features(source_df, target_df):
    df_f = target_df.copy()
    
    # 1. Raw Autoregressive Lags
    lag_df = source_df[['geohash', 'day', 'timestamp', 'demand']].copy()
    
    lag_24 = lag_df.copy()
    lag_24['day'] += 1
    lag_24.rename(columns={'demand': 'lag_24h'}, inplace=True)
    
    lag_48 = lag_df.copy()
    lag_48['day'] += 2
    lag_48.rename(columns={'demand': 'lag_48h'}, inplace=True)
    
    df_f = df_f.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
    df_f = df_f.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')
    df_f[['lag_24h', 'lag_48h']] = df_f[['lag_24h', 'lag_48h']].fillna(0.0)
    
    # 2. Raw Temporal Momentum
    df_f['momentum_24_48'] = df_f['lag_24h'] - df_f['lag_48h']
    
    # 3. Raw Spatial Spillover Mapping
    lag_24_lookup = lag_24.set_index(['geohash', 'day', 'timestamp'])['lag_24h'].to_dict()
    
    def compute_raw_spillover(row):
        try:
            neighbors = pgh.neighbors(row['geohash'])
            spillover_sum = 0.0
            valid_nodes = 0
            for n in neighbors:
                key = (n, row['day'], row['timestamp'])
                if key in lag_24_lookup:
                    spillover_sum += lag_24_lookup[key]
                    valid_nodes += 1
            return spillover_sum / valid_nodes if valid_nodes > 0 else 0.0
        except:
            return 0.0
            
    print("Computing Raw Adjacency Spillover Graph...")
    df_f['neighbor_spillover_24h'] = df_f.apply(compute_raw_spillover, axis=1)
    
    # 4. Kinematics & Interaction Keys
    t_split = df_f['timestamp'].str.split(':', expand=True).astype(int)
    ts_minutes = t_split[0] * 60 + t_split[1]
    df_f['hour'] = t_split[0]
    df_f['time_slot_15m'] = ts_minutes // 15
    df_f['hour_sin'] = np.sin(2 * np.pi * df_f['hour'] / 24.0)
    df_f['hour_cos'] = np.cos(2 * np.pi * df_f['hour'] / 24.0)
    df_f['is_rush_hour'] = df_f['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df_f['geo_time_interaction'] = df_f['geohash'].astype(str) + "_" + df_f['time_slot_15m'].astype(str)
    
    df_f['Temperature'] = df_f['Temperature'].fillna(df_f['Temperature'].median())
    for col in ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']:
        if col in df_f.columns:
            df_f[col] = df_f[col].fillna('Unknown')
            
    return df_f

X_train_fe = engineer_v15_features(raw_train, raw_train.drop(columns=['demand'], errors='ignore'))
X_test_fe = engineer_v15_features(raw_train, raw_test.drop(columns=['Index'], errors='ignore'))

# 5. OOF Target Encoding (The Baseline)
def apply_oof_encoding(tr_df, te_df, tgt, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    tr_enc = np.zeros(len(tr_df))
    tmp_tr = tr_df[[col]].copy()
    tmp_tr['tgt'] = tgt
    g_mean = tgt.mean()
    
    for tr_idx, val_idx in kf.split(tmp_tr):
        f_map = tmp_tr.iloc[tr_idx].groupby(col)['tgt'].mean()
        tr_enc[val_idx] = tmp_tr.iloc[val_idx][col].map(f_map).fillna(g_mean).values
        
    te_map = tmp_tr.groupby(col)['tgt'].mean()
    te_enc = te_df[col].map(te_map).fillna(g_mean).values
    return tr_enc, te_enc

X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geo_time_interaction')
X_train_fe['TE_geohash'], X_test_fe['TE_geohash'] = apply_oof_encoding(X_train_fe, X_test_fe, y_train, 'geohash')

# V15 Upgrade: Weather x Baseline Interaction Multiplier
# We label encode Weather first, then multiply it by the baseline to scale environmental risk
le_weather = LabelEncoder()
le_weather.fit(X_train_fe['Weather'].astype(str).tolist() + X_test_fe['Weather'].astype(str).tolist())
X_train_fe['Weather_Encoded'] = le_weather.transform(X_train_fe['Weather'].astype(str))
X_test_fe['Weather_Encoded'] = le_weather.transform(X_test_fe['Weather'].astype(str))

X_train_fe['Weather_Risk_Multiplier'] = X_train_fe['TE_geo_time'] * (X_train_fe['Weather_Encoded'] + 1)
X_test_fe['Weather_Risk_Multiplier'] = X_test_fe['TE_geo_time'] * (X_test_fe['Weather_Encoded'] + 1)

cat_features = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_features:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

drop_columns = ['timestamp', 'geo_time_interaction', 'Index', 'Weather_Encoded']
features = [c for c in X_train_fe.columns if c not in drop_columns]

X = X_train_fe[features].values
X_test = X_test_fe[features].values
X_test = np.nan_to_num(X_test, nan=0.0)

print(f"Matrix formatting completed. Shape: {X.shape}")

Computing Raw Adjacency Spillover Graph...
Computing Raw Adjacency Spillover Graph...
Matrix formatting completed. Shape: (77299, 20)


### Phase 3: Dynamic Asymmetric Engine Compilation
Executes the 5-Fold validation Triad. Crucially, the LightGBM objective is overridden with a Dynamic Asymmetric Mean Squared Error gradient. The gradient penalty for missing a traffic spike scales continuously based on the underlying severity of the true demand. This forces the engine to aggressively protect high-variance gridlock peaks while preventing mathematical over-correction in quiet neighborhoods.

In [3]:
# V15 Upgrade: Dynamic Asymmetric MSE
# The penalty for missing a traffic jam scales linearly with the severity of the true demand
def dynamic_asymmetric_mse(preds, train_data):
    y_true = train_data.get_label()
    residual = (y_true - preds).astype("float")
    
    # If y_true is 0.9, multiplier is 1.0 + (0.9 * 0.8) = 1.72x penalty
    # If y_true is 0.1, multiplier is 1.0 + (0.1 * 0.8) = 1.08x penalty
    penalty_multiplier = 1.0 + (y_true * 0.8)
    
    grad = np.where(residual > 0, -2.0 * penalty_multiplier * residual, -2.0 * residual)
    hess = np.where(residual > 0, 2.0 * penalty_multiplier, 2.0)
    return grad, hess

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Dropped learning rate slightly to let the trees optimize the dynamic penalty safely
lgb_params = {'objective': dynamic_asymmetric_mse, 'metric': 'rmse', 'learning_rate': 0.025, 'max_depth': 8, 'num_leaves': 128, 'min_child_samples': 20, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.025, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1}
cat_params = {'iterations': 3000, 'learning_rate': 0.025, 'depth': 8, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("Initiating V15 Dynamic Asymmetric Triad Training...")

for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X[t_idx], y_train[t_idx]
    X_va, y_va = X[v_idx], y_train[v_idx]
    
    m_lgb = lgb.train(lgb_params, lgb.Dataset(X_tr, y_tr), num_boost_round=3000, valid_sets=[lgb.Dataset(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[v_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / 5
    
    m_xgb = xgb.train(xgb_params, xgb.DMatrix(X_tr, y_tr), 3000, evals=[(xgb.DMatrix(X_va, y_va), 'val')], early_stopping_rounds=150, verbose_eval=False)
    oof_xgb[v_idx] = m_xgb.predict(xgb.DMatrix(X_va))
    test_xgb += m_xgb.predict(xgb.DMatrix(X_test)) / 5
    
    m_cat = CatBoostRegressor(**cat_params).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[v_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / 5
    
    print(f"Fold {fold+1} validation finalized.")

Initiating V15 Dynamic Asymmetric Triad Training...
Fold 1 validation finalized.
Fold 2 validation finalized.
Fold 3 validation finalized.
Fold 4 validation finalized.
Fold 5 validation finalized.


### Phase 4: Drift-Resistant Convergence & Assembly
Evaluates the Out-of-Fold prediction arrays using a SciPy Nelder-Mead optimization simplex. This extracts a generalized, drift-resistant ensemble blend, applies strict physical capacity bounds `[0.0, 1.0]`, and generates the final evaluation matrix.

In [4]:
def optimization_objective(weights):
    weights = np.array(weights)
    if weights.sum() == 0: return 999.0
    normalized_weights = weights / weights.sum()
    blended_prediction = (normalized_weights[0] * oof_lgb) + (normalized_weights[1] * oof_xgb) + (normalized_weights[2] * oof_cat)
    return -max(0, 100 * r2_score(y_train, blended_prediction))

optimal_weights = minimize(optimization_objective, [0.40, 0.30, 0.30], method='Nelder-Mead').x
optimal_weights /= sum(optimal_weights)

final_test_predictions = np.clip((optimal_weights[0] * test_lgb) + (optimal_weights[1] * test_xgb) + (optimal_weights[2] * test_cat), 0.0, 1.0)
final_r2 = max(0, 100 * r2_score(y_train, (optimal_weights[0] * oof_lgb) + (optimal_weights[1] * oof_xgb) + (optimal_weights[2] * oof_cat)))

submission_payload = pd.DataFrame({'Index': submission_index, 'demand': final_test_predictions})
submission_payload.to_csv("submission_v15.csv", index=False)

print("\n==================================================")
print("PIPELINE EXECUTION COMPLETE (V15 PERFECTED GRAPH)")
print("==================================================")
print(f"Optimal Engine Distribution: LGBM: {optimal_weights[0]:.3f} | XGB: {optimal_weights[1]:.3f} | CAT: {optimal_weights[2]:.3f}")
print(f"Terminal R2 Score (The Legitimate Target): {final_r2:.4f}")
print("Output Matrix Generated: submission_v15.csv")


PIPELINE EXECUTION COMPLETE (V15 PERFECTED GRAPH)
Optimal Engine Distribution: LGBM: 0.105 | XGB: 0.595 | CAT: 0.300
Terminal R2 Score (The Legitimate Target): 94.9594
Output Matrix Generated: submission_v15.csv
